# Мерей: оптимизация шагов на основе решения Анаис

Неисполненный шаблон. Текущий Мерей: 766d0a39; источник Анаис: 78a2ec99.
Выбранный runtime commit задаётся ниже. Результаты появятся после выполнения
всех ячеек; заранее подтверждённого ускорения этот notebook не содержит.

Основное сравнение: current Мерей → selected, два блока ABBA по умолчанию
(восемь запусков, по четыре на каждое решение).
Сравниваются исходные метрики harness: полное время, цикл обучения и сумма
времени шагов 3–64. Native Soup используется отдельно для official admission.
Две серии сохраняются раздельно.

Kaggle: GPU T4 x2, Internet On, свежая сессия, затем Run All. Исходный harness
использует одну T4; все training children запускаются последовательно.
Пик selected должен быть ≤5175 MiB в обеих сериях и не выше current в A/B.
Модель, NF4, LoRA, seed, данные, порядок, batch size и 64 шага определяет
исходный task; здесь настройки обучения не меняются.

Зависимости закрепляются до Transformers/TRL imports. Затем пакеты и metadata
не меняются. Native setup-prefix проверяется отдельным CPU child со скрытой
GPU; training получает исходное окружение harness. Controller не импортирует
Torch и не создаёт CUDA-контекст. Новые timers и memory monitors не вводятся.

Checkout сохраняет исходные bytes, включая CRLF; каждый solution blob
сверяется с Git. Notebook и evidence находятся вне solution/. Атрибуция
Анаис, Нұржігіта, Владимира, Мерея и организаторов сохраняется в исходниках.


In [ ]:
import hashlib, importlib.metadata as metadata, json, os
import shutil, statistics, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/promaladoi/Decentrathon-6.0.git'
CANDIDATE = 'ce2c4e4ab547088abd17dde87997e16f798a4695'
PREVIOUS_MEREI = '766d0a39fe8394be8a1cd280401af1adde6d5206'
ANAIS_SOURCE = '78a2ec998ca7b665d1aa70299aeb6f2f2a105a57'
assert len(CANDIDATE) == 40 and all(c in '0123456789abcdef' for c in CANDIDATE), 'Fill selected commit'
MEMORY_CEILING_MIB = 5175
WORKING = Path('/kaggle/working')
EVIDENCE = WORKING / ('merey-stage3-' + CANDIDATE[:12] + '-evidence')
EVIDENCE.mkdir(exist_ok=False)

gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpu_names, flush=True)
assert 'T4' in gpu_names, 'Choose GPU T4 x2'

ABBA_ROUNDS = 2  # Two complete ABBA blocks: four runs per variant.
assert type(ABBA_ROUNDS) is int and ABBA_ROUNDS >= 2


In [ ]:
def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# The controller uses distribution metadata, not imports of these libraries.
# Finish every install/removal here before any test or training child starts.
assert installed('torch') == '2.11.0+cu128', 'Use the recorded Kaggle image'
assert installed('triton') == '3.6.0', 'Use Triton 3.6.0 from the recorded image'
requirements = {
    'soup-cli': ('0.77.0', 'soup-cli[train]==0.77.0'),
    'transformers': ('5.16.1', 'transformers==5.16.1'),
    'peft': ('0.20.0', 'peft==0.20.0'),
    'trl': ('0.29.1', 'trl==0.29.1'),
    'bitsandbytes': ('0.50.2', 'bitsandbytes==0.50.2'),
    'accelerate': ('1.14.0', 'accelerate==1.14.0'),
    'datasets': ('4.8.5', 'datasets==4.8.5'),
}
missing = [spec for name, (required, spec) in requirements.items()
           if installed(name) != required]
if installed('pytest') is None:
    missing.append('pytest')
if installed('torchao') is not None:
    # Remove the optional incompatible package before the fixed environment begins.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
versions = {name: installed(name) for name in ['torch', 'triton', *requirements, 'pytest']}
assert all(versions[name] == required for name, (required, spec) in requirements.items())
assert versions['torch'] == '2.11.0+cu128' and versions['triton'] == '3.6.0'
print(json.dumps(versions, indent=2), flush=True)
# No later cell installs, removes, upgrades or edits packages.

PACKAGES_FIXED = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(EVIDENCE / 'fixed-packages.txt').write_text(PACKAGES_FIXED, encoding='utf-8')


In [ ]:
def clone_exact(label, commit):
    checkout = WORKING / ('merey-stage3-' + label + '-' + commit[:12])
    assert not checkout.exists(), 'Preserve previous results and use a fresh session'
    subprocess.run(['git', '-c', 'core.autocrlf=false', 'clone', '-q', '--no-checkout', REPO_URL, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'config', 'core.autocrlf', 'false'], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', '--detach', commit], check=True)
    actual = subprocess.check_output(
        ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == commit
    return checkout / 'Мерей'

roots = {'candidate': clone_exact('candidate', CANDIDATE),
         'previous': clone_exact('previous', PREVIOUS_MEREI)}
for directory in ['harness', 'task', 'tests']:
    a, b = roots['previous'] / directory, roots['candidate'] / directory
    files = sorted(p.relative_to(a) for p in a.rglob('*') if p.is_file())
    assert files == sorted(p.relative_to(b) for p in b.rglob('*') if p.is_file())
    assert all((a / p).read_bytes() == (b / p).read_bytes() for p in files), directory

sys.path.insert(0, str(roots['candidate'] / 'harness'))
from common import ENV
assert ENV['CUDA_VISIBLE_DEVICES'] == '0', 'Use the original one-GPU harness environment'
assert ENV['PYTORCH_CUDA_ALLOC_CONF'] == 'expandable_segments:True'

def run_repo(label, arguments, log_name, **overrides):
    print('RUN_BEGIN', label, arguments, flush=True)
    log = EVIDENCE / log_name
    with log.open('w', encoding='utf-8') as stream:
        result = subprocess.run([sys.executable, *arguments], cwd=roots[label],
                                env=dict(ENV, **overrides), stdout=stream,
                                stderr=subprocess.STDOUT)
    output = log.read_text(encoding='utf-8')
    print(output, flush=True)
    result.check_returncode()
    return output

def solution_hash(root, python_only=False):
    digest = hashlib.sha256()
    for path in sorted((root / 'solution').rglob('*')):
        if (path.is_file() and '__pycache__' not in path.parts
                and (not python_only or path.suffix == '.py')):
            digest.update(path.relative_to(root / 'solution').as_posix().encode())
            digest.update(path.read_bytes())
    return digest.hexdigest()

measured_solution_git_tree = subprocess.check_output(
    ['git', '-C', str(roots['candidate'].parent), 'rev-parse',
     CANDIDATE + ':Мерей/solution'], text=True).strip()
assert len(measured_solution_git_tree) == 40

provenance = {
    'candidate_runtime_commit': CANDIDATE,
    'previous_merei_runtime_commit': PREVIOUS_MEREI,
    'versions': versions,
    'candidate_solution_sha256_before_tests': solution_hash(roots['candidate']),
    'candidate_python_sha256': solution_hash(roots['candidate'], python_only=True),
    'measured_solution_git_tree': measured_solution_git_tree,
    'memory_ceiling_mib': MEMORY_CEILING_MIB,
    'anais_source_commit': ANAIS_SOURCE,
    'anais_source_path': 'Анаис/solution',
}
provenance['previous_solution_sha256'] = solution_hash(roots['previous'])
(EVIDENCE / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print(json.dumps(provenance, indent=2), flush=True)


In [ ]:
def verify_solution_git_blobs(root, commit):
    tree = subprocess.check_output(
        ['git', '-C', str(root.parent), 'rev-parse', commit + ':Мерей/solution'], text=True).strip()
    listing = subprocess.check_output(['git', '-C', str(root.parent), 'ls-tree', '-rz', tree])
    records = []
    for entry in listing.split(b'\0'):
        if not entry:
            continue
        descriptor, raw_path = entry.split(b'\t', 1)
        mode, kind, expected = descriptor.decode().split()
        assert kind == 'blob' and mode == '100644'
        path = raw_path.decode('utf-8')
        raw = (root / 'solution' / path).read_bytes()
        actual = hashlib.sha1(b'blob ' + str(len(raw)).encode() + b'\0' + raw).hexdigest()
        assert actual == expected, 'Checkout bytes differ from Git: ' + path
        records.append({'path': path, 'git_blob_sha': actual,
                        'sha256': hashlib.sha256(raw).hexdigest(), 'bytes': len(raw)})
    assert records
    return {'commit': commit, 'solution_git_tree': tree, 'files': records,
            'solution_sha256': solution_hash(root),
            'python_sha256': solution_hash(root, python_only=True)}

source_proofs = {label: verify_solution_git_blobs(roots[label], commit)
                 for label, commit in [('previous', PREVIOUS_MEREI), ('candidate', CANDIDATE)]}
for directory in ['harness', 'task', 'tests']:
    trees = [subprocess.check_output(
        ['git', '-C', str(roots[label].parent), 'rev-parse', commit + ':Мерей/' + directory],
        text=True).strip() for label, commit in [('previous', PREVIOUS_MEREI), ('candidate', CANDIDATE)]]
    assert trees[0] == trees[1], directory + ' must remain unchanged'
provenance['source_proofs'] = source_proofs
(EVIDENCE / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')


## Native parity, tests и исходные данные

CPU diagnostic вызывает настоящий Soup setup-prefix и доказывает native
metadata parity, восстановление функции и отсутствие model/seed/CUDA work.
Только его отдельный child получает пустой CUDA_VISIBLE_DEVICES.
Затем исполняются полный solution suite и все 79 Soup regressions с решением.
Counts solution suite не зашиты: фактические passed/skipped сохраняются в логах.
task/prepare_data.py запускается без изменений, bytes данных сравниваются.


In [ ]:
parity_output = run_repo(
    'candidate', ['solution/test_package_metadata_snapshot.py', 'NativeInstalledTests'],
    'native-metadata-parity.log',
    MEREY_NATIVE_METADATA_SNAPSHOT_PARITY='1', CUDA_VISIBLE_DEVICES='')
prefix = 'native metadata snapshot parity: '
records = [json.loads(line.split(prefix, 1)[1])
           for line in parity_output.splitlines() if prefix in line]
assert len(records) == 1, 'The explicit installed-native parity diagnostic must run'
parity = records[0]
assert parity['native_mapping_equal'] and parity['independent_dict_and_lists']
assert parity['function_restored'] and not parity['cuda_initialized']
assert parity['cpu_probe_gpu_hidden']
assert parity['setup_calls'] == {'wrapper': 1, 'native': 1}
assert parity['actual_snapshot_returns_to_trl'] == 1
assert parity['native_scan_callers_during_setup'] == []
assert parity['stopped_before_seed_and_model'] and parity['setup_idempotent']
(EVIDENCE / 'native-metadata-parity.json').write_text(
    json.dumps(parity, indent=2), encoding='utf-8')


In [ ]:
run_repo('candidate', ['-m', 'pytest', 'solution', '-q', '-p', 'no:cacheprovider'],
         'solution-tests.log')
run_repo('candidate', ['-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'],
         'soup-regressions.log', SOUP_APPLY_SOLUTION='1')
for label in roots:
    run_repo(label, ['task/prepare_data.py'], label + '-prepare-data.log')
assert solution_hash(roots['candidate']) == provenance['candidate_solution_sha256_before_tests']

assert '79 passed' in (EVIDENCE / 'soup-regressions.log').read_text(encoding='utf-8')
for filename in ['train.jsonl', 'eval_public.jsonl']:
    previous = roots['previous'] / 'data' / filename
    candidate = roots['candidate'] / 'data' / filename
    assert previous.read_bytes() == candidate.read_bytes(), 'Task data/order must match'
    destination = EVIDENCE / 'task-data' / filename
    destination.parent.mkdir(exist_ok=True)
    shutil.copyfile(candidate, destination)
for label in roots:
    assert solution_hash(roots[label]) == source_proofs[label]['solution_sha256']


## Current Мерей → selected: два блока ABBA

Исходный warm_up загружает модель до измерений. Каждый training child
запускается неизменённым run_task.train с native task, seed, cold compiler
caches, внешним полным timer, memory poller и callback metrics. Новые
training timers не вставляются. Между training children нет лишнего eval.
Качество проверяет отдельная official серия ниже.

timed_loop_s — сумма исходных step timings со шага 3. train_loop_s и wall_s
являются отдельными метриками. По умолчанию выполняются два полных блока
ABBA: восемь raw результатов, по четыре на каждое решение. Сохраняются
все per-run metrics/logs и разброс каждого решения. Положительный процент
означает сокращение времени; большее число повторов снижает влияние шума.


In [ ]:
warmup_child = "\n".join([
    "import sys", "from pathlib import Path",
    "sys.path.insert(0, str(Path.cwd() / 'harness'))",
    "from run_task import warm_up",
    "out = Path(sys.argv[1]); out.mkdir(parents=True, exist_ok=False)",
    "warm_up(out)", ""])
for label in roots:
    run_repo(label, ['-c', warmup_child, 'runs_stage3_ab_warmup'], 'ab-' + label + '-warmup.log')

training_child = "\n".join([
    "import sys", "from pathlib import Path", "from types import SimpleNamespace",
    "sys.path.insert(0, str(Path.cwd() / 'harness'))",
    "from common import REPO, write_json",
    "from run_task import load_task, train",
    "name = sys.argv[1]",
    "out = REPO / 'runs_stage3_previous_ab'; out.mkdir(exist_ok=True)",
    "task = load_task()",
    "args = SimpleNamespace(out=out, train_file=REPO/'data'/'train.jsonl',",
    "                       eval_file=REPO/'data'/'eval_public.jsonl',",
    "                       seed=task['training']['seed'], timeout=900)",
    "result = train('solution', name, args)",
    "write_json(out / (name + '-result.json'), result)",
    "assert result['exit_code'] == 0",
    "assert result['global_step'] == 64 and result['metrics']['steps'] == 64",
    "assert result['metrics']['timed_steps'] == 62",
    "print('NATIVE_AB_TRAIN_COMPLETE', name, flush=True)", ""])
(EVIDENCE / 'ab-training-controller.py').write_text(training_child, encoding='utf-8')
order = []
for block in range(ABBA_ROUNDS):
    first = 2 * block + 1
    order.extend([('previous', 'old_' + str(first)), ('candidate', 'new_' + str(first)),
                  ('candidate', 'new_' + str(first + 1)), ('previous', 'old_' + str(first + 1))])
assert len(order) == 4 * ABBA_ROUNDS and len({name for label, name in order}) == len(order)
ab_runs = {'previous': [], 'candidate': []}
for label, name in order:
    run_repo(label, ['-c', training_child, name], 'ab-' + name + '.log')
    path = roots[label] / 'runs_stage3_previous_ab' / (name + '-result.json')
    run = json.loads(path.read_text())
    assert run['exit_code'] == 0
    assert run['metrics']['steps'] == 64 and run['metrics']['timed_steps'] == 62
    assert run['gpu_start_mib'] <= 1000, 'A/B GPUs must be idle at each run start'
    assert run['metrics']['visible_devices'] == '0' and run['metrics']['device_count'] == 1
    assert run['metrics']['alloc_conf'] == 'expandable_segments:True'
    ab_runs[label].append(run)
    shutil.copyfile(path, EVIDENCE / ('ab-' + name + '-result.json'))

def summarize_current(rows):
    return {
        'wall_runs_s': [r['wall_s'] for r in rows],
        'wall_median_s': statistics.median(r['wall_s'] for r in rows),
        'gpu_peaks_mib': [r['gpu_peak_mib'] for r in rows],
        'gpu_peak_mib': max(r['gpu_peak_mib'] for r in rows),
        'train_loop_runs_s': [r['metrics']['train_loop_s'] for r in rows],
        'train_loop_median_s': statistics.median(r['metrics']['train_loop_s'] for r in rows),
        'timed_steps_3_to_64_runs_s': [r['metrics']['timed_loop_s'] for r in rows],
        'timed_steps_3_to_64_median_s': statistics.median(r['metrics']['timed_loop_s'] for r in rows),
    }
ab_summary = {label: summarize_current(rows) for label, rows in ab_runs.items()}
assert all(len(rows) == 2 * ABBA_ROUNDS for rows in ab_runs.values())
ab_comparison = {
    'kind': 'current_merei_vs_selected_abba_not_official_admission',
    'previous_commit': PREVIOUS_MEREI, 'candidate_commit': CANDIDATE,
    'abba_rounds': ABBA_ROUNDS, 'runs_per_variant': 2 * ABBA_ROUNDS,
    'order': [{'block': index // 4 + 1, 'label': label, 'run': name}
              for index, (label, name) in enumerate(order)], 'summary': ab_summary,
}
for label, metric in [('whole_command', 'wall_median_s'), ('training_loop', 'train_loop_median_s'),
                      ('timed_steps_3_to_64', 'timed_steps_3_to_64_median_s')]:
    ab_comparison[label + '_time_reduction_pct'] = 100. * (
        1. - ab_summary['candidate'][metric] / ab_summary['previous'][metric])
ab_comparison['actual_steps_faster'] = ab_comparison['timed_steps_3_to_64_time_reduction_pct'] > 0
ab_comparison['candidate_memory_within_ceiling'] = ab_summary['candidate']['gpu_peak_mib'] <= MEMORY_CEILING_MIB
ab_comparison['candidate_memory_not_above_current'] = (
    ab_summary['candidate']['gpu_peak_mib'] <= ab_summary['previous']['gpu_peak_mib'])
(EVIDENCE / 'previous-merei-ab.json').write_text(json.dumps(ab_comparison, indent=2), encoding='utf-8')
print(json.dumps(ab_comparison, indent=2), flush=True)
assert ab_comparison['candidate_memory_within_ceiling'], 'A/B candidate exceeds 5175 MiB'
assert ab_comparison['candidate_memory_not_above_current'], 'A/B candidate increases current memory'


## Отдельный official admission: два повтора и десять правил

Неизменённый official harness выполняет base → solution → solution → base,
eval и output checks. check_rules.py выполняет все десять проверок, включая
79 Soup regressions. Эта серия не смешивается с current-Мерей A/B.
Предел 5175 MiB проверяется для каждого solution run. Исходный checks SHA
связывается с точными измеренными solution bytes. Пакеты и source неизменны.


In [ ]:
ADMISSION = 'runs_reproduce_stage3_admission'
run_repo('candidate', ['harness/run_task.py', '--repeats', '2', '--out', ADMISSION],
         'official-training.log')
run_repo('candidate', ['harness/check_rules.py', '--result', ADMISSION + '/result.json'],
         'official-checks.log')
result_path = roots['candidate'] / ADMISSION / 'result.json'
checks_path = roots['candidate'] / ADMISSION / 'checks.json'
result = json.loads(result_path.read_text())
checks = json.loads(checks_path.read_text())
assert len(checks['checks']) == 10 and all(item['ok'] for item in checks['checks'])
assert checks['result_sha256'] == hashlib.sha256(result_path.read_bytes()).hexdigest()
assert checks['solution_sha256'] == solution_hash(roots['candidate'])
assert checks['solution_sha256'] == provenance['candidate_solution_sha256_before_tests']
candidate = result['summary']['solution']
assert candidate['gpu_peak_mib'] <= MEMORY_CEILING_MIB, 'Stage3 memory ceiling exceeded'
shutil.copy2(result_path, EVIDENCE / 'final-result.json')
shutil.copy2(checks_path, EVIDENCE / 'final-checks.json')
print('ADMISSION_COMPLETE', json.dumps({
    'candidate_runtime_commit': CANDIDATE, 'candidate_summary': candidate,
    'checks_passed': 10, 'memory_ceiling_mib': MEMORY_CEILING_MIB,
}, indent=2), flush=True)

selected_runs = result['runs']['solution']
assert len(selected_runs) == 2
assert all(r['exit_code'] == 0 and r['global_step'] == 64 and r['metrics']['steps'] == 64
           and r['metrics']['timed_steps'] == 62 for r in selected_runs)
assert all(r['gpu_peak_mib'] <= MEMORY_CEILING_MIB for r in selected_runs)
assert result['packages_changed'] == []
(EVIDENCE / 'official-selected-summary.json').write_text(json.dumps({
    'candidate_commit': CANDIDATE, 'series': 'official_admission_separate_from_current_ab',
    'summary': result['summary']['solution'], 'step_summary': summarize_current(selected_runs),
    'checks_passed': 10}, indent=2), encoding='utf-8')


## Evidence и неизменность исходников

Архив включает точные solution bytes current/selected, Git blob SHA/SHA256,
логи тестов и setup-prefix, task data, все A/B результаты, original official
result/checks и небольшие per-run артефакты. Адаптеры и compiler caches
не входят в компактный архив; native run directories сохраняются отдельно
в Kaggle output. Manifest проверяет bytes каждого файла.

Финальная acceptance требует ускорения шагов 3–64, соблюдения памяти обеих
серий и 10/10 checks. Если A/B не подтверждает ускорение, evidence сохраняется,
но notebook не объявляет успешную acceptance.


In [ ]:
def collect_small_run_artifacts(label, run_folder, relative):
    source = roots[label] / run_folder
    target = EVIDENCE / 'runs' / relative
    allowed = {'metrics.json', 'task.yaml', 'train.log', 'warmup.log',
               'trainer_state.json', 'rewind.jsonl', 'adapter_config.json',
               'config.json', 'training_args.json'}
    for path in sorted(source.rglob('*')):
        if not path.is_file() or any(p in ('triton_cache', 'inductor_cache') for p in path.parts):
            continue
        if path.name in allowed or (path.parent == source and path.suffix in ('.json', '.log')):
            destination = target / path.relative_to(source)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, destination)

for label, name in order:
    collect_small_run_artifacts(label, 'runs_stage3_previous_ab/' + name, 'ab/' + name)
collect_small_run_artifacts('candidate', ADMISSION, 'official')
for label, commit in [('previous', PREVIOUS_MEREI), ('candidate', CANDIDATE)]:
    actual = verify_solution_git_blobs(roots[label], commit)
    assert actual == source_proofs[label]
    subprocess.run(['git', '-C', str(roots[label].parent), 'diff', '--exit-code', '--',
                    'Мерей/solution', 'Мерей/harness', 'Мерей/task', 'Мерей/tests'], check=True)
    for record in actual['files']:
        source = roots[label] / 'solution' / record['path']
        destination = EVIDENCE / 'sources' / label / 'solution' / record['path']
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)
assert subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True) == PACKAGES_FIXED
acceptance = {
    'candidate_commit': CANDIDATE, 'previous_merei_commit': PREVIOUS_MEREI,
    'anais_source_commit': ANAIS_SOURCE,
    'abba_rounds': ABBA_ROUNDS, 'ab_runs_per_variant': 2 * ABBA_ROUNDS,
    'actual_steps_faster_in_current_ab': ab_comparison['actual_steps_faster'],
    'current_ab_timed_steps_3_to_64_reduction_pct': ab_comparison['timed_steps_3_to_64_time_reduction_pct'],
    'current_ab_candidate_peak_mib': ab_summary['candidate']['gpu_peak_mib'],
    'official_candidate_peak_mib': result['summary']['solution']['gpu_peak_mib'],
    'memory_ceiling_mib': MEMORY_CEILING_MIB,
    'current_ab_memory_not_above_previous': ab_comparison['candidate_memory_not_above_current'],
    'official_checks_passed': sum(item['ok'] for item in checks['checks']),
    'solution_sha256': solution_hash(roots['candidate']),
    'solution_git_tree': source_proofs['candidate']['solution_git_tree'],
    'source_unchanged': True, 'packages_unchanged': True,
}
acceptance['accepted'] = (
    acceptance['actual_steps_faster_in_current_ab']
    and acceptance['current_ab_memory_not_above_previous']
    and acceptance['current_ab_candidate_peak_mib'] <= MEMORY_CEILING_MIB
    and acceptance['official_candidate_peak_mib'] <= MEMORY_CEILING_MIB
    and acceptance['official_checks_passed'] == 10)
(EVIDENCE / 'acceptance.json').write_text(json.dumps(acceptance, indent=2), encoding='utf-8')
manifest = {'kind': 'sha256_exact_evidence_bytes', 'files': []}
for path in sorted(EVIDENCE.rglob('*')):
    if path.is_file() and path.name != 'manifest.json':
        raw = path.read_bytes()
        manifest['files'].append({'path': path.relative_to(EVIDENCE).as_posix(), 'bytes': len(raw),
                                  'sha256': hashlib.sha256(raw).hexdigest()})
(EVIDENCE / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
for item in manifest['files']:
    raw = (EVIDENCE / item['path']).read_bytes()
    assert len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256']
archive = Path(shutil.make_archive(str(EVIDENCE), 'zip', root_dir=EVIDENCE))
receipt = {'archive': str(archive), 'bytes': archive.stat().st_size,
           'sha256': hashlib.sha256(archive.read_bytes()).hexdigest(), 'acceptance': acceptance}
EVIDENCE.with_name(EVIDENCE.name + '-receipt.json').write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print(json.dumps(receipt, indent=2), flush=True)
print('EVIDENCE_DIRECTORY', str(EVIDENCE), flush=True)
assert acceptance['accepted'], 'Evidence saved; current A/B does not establish the required step improvement'
print('MEREY_STAGE3_REPRODUCTION_COMPLETE', flush=True)
